Tasks
- Add the standard scaler step: see any difference
- Add  the evaluation part that can show rmse, mae, r2
- Save the best model

In [ ]:
# Installing required packages
!pip install pyspark
!pip install findspark

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SQLContext

In [ ]:
spark = SparkSession \
    .builder \
    .appName("ML_test") \
    .getOrCreate()

# Create Spark Context

In [ ]:
sc = spark.sparkContext
sqlContext = SQLContext(sc)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving diabetes.csv to diabetes.csv


use diabetes

In [ ]:
file='diabetes.csv'

df = sqlContext.read.load(file,
                          format='com.databricks.spark.csv',
                          header='true',inferSchema='true')

#  Regression

In [ ]:
from pyspark.sql import DataFrameNaFunctions
from pyspark.ml.feature import VectorAssembler , StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline


In [ ]:
df = df.na.drop()

In [ ]:
df.columns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

feature columns = Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age

In [ ]:
featureColumns =df.columns[0:-1]

In [ ]:
featureColumns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

In [ ]:
df=df.withColumnRenamed("Outcome","label")

In [ ]:
(trainingData, testData) = df.randomSplit([0.8,0.2], seed = 13234 )

label = what we wanna predict based on features

What is Scaler? :
- Some machine learning models (like logistic regression) don’t work well if features (columns) have different scales because they may give too much importance to bigger numbers, even if the feature is not more important but the model thinks it is
- StandardScaler fixes this by doing two things to each column:
  - Subtract the mean → center the values around 0
  - Divide by the standard deviation → make the spread of values similar
  - You now have new values centered around 0, with similar spread
  - The Scaler function will transform the data into a new column called "scaledFeatures" where each feature is now standardized

In [ ]:
assembler = VectorAssembler(inputCols=featureColumns, outputCol="features")
scaler= StandardScaler(inputCol= "features", outputCol= "scaled_features")
lr= LinearRegression(featuresCol= "scaled_features")

In [ ]:
pipeline = Pipeline(stages=[assembler,scaler, lr])

In [ ]:
model = pipeline.fit(trainingData)

In [ ]:
prediction = model.transform(testData)

In [ ]:
prediction.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-----+--------------------+--------------------+--------------------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|label|            features|     scaled_features|          prediction|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-----+--------------------+--------------------+--------------------+
|          0|     91|           80|            0|      0|32.4|                   0.601| 27|    0|[0.0,91.0,80.0,0....|[0.0,2.8287003815...| 0.05657670865193232|
|          0|     95|           85|           25|     36|37.4|                   0.247| 24|    1|[0.0,95.0,85.0,25...|[0.0,2.9530388598...|  0.1212915585776484|
|          0|     98|           82|           15|     84|25.2|                   0.299| 22|    0|[0.0,98.0,82.0,15...|[0.0,3.0462927185...|-0.03530504517640132|
|          0|    100|           70

In [ ]:
prediction.show(truncate=False)

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-----+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin|BMI |DiabetesPedigreeFunction|Age|label|features                                   |scaled_features                                                                                                                          |prediction          |
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-----+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|0          |91     |80           |0            |0      |32.4|0.601                   |27 |0    |

In [ ]:
prediction.toPandas()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,label,features,scaled_features,prediction
0,0,91,80,0,0,32.4,0.601,27,0,"[0.0, 91.0, 80.0, 0.0, 0.0, 32.4, 0.601, 27.0]","[0.0, 2.8287003815542695, 4.113551130832826, 0...",0.056577
1,0,95,85,25,36,37.4,0.247,24,1,"[0.0, 95.0, 85.0, 25.0, 36.0, 37.4, 0.247, 24.0]","[0.0, 2.953038859864347, 4.370648076509878, 1....",0.121292
2,0,98,82,15,84,25.2,0.299,22,0,"[0.0, 98.0, 82.0, 15.0, 84.0, 25.2, 0.299, 22.0]","[0.0, 3.0462927185969053, 4.216389909103647, 0...",-0.035305
3,0,100,70,26,50,30.8,0.597,21,0,"[0.0, 100.0, 70.0, 26.0, 50.0, 30.8, 0.597, 21.0]","[0.0, 3.108461957751944, 3.5993572394787225, 1...",0.114929
4,0,101,64,17,0,21.0,0.252,21,0,"[0.0, 101.0, 64.0, 17.0, 0.0, 21.0, 0.252, 21.0]","[0.0, 3.1395465773294635, 3.290840904666261, 1...",-0.028391
...,...,...,...,...,...,...,...,...,...,...,...,...
149,13,106,70,0,0,34.2,0.251,52,0,"[13.0, 106.0, 70.0, 0.0, 0.0, 34.2, 0.251, 52.0]","[3.883025612848556, 3.294969675217061, 3.59935...",0.542010
150,13,106,72,54,0,36.6,0.178,45,0,"[13.0, 106.0, 72.0, 54.0, 0.0, 36.6, 0.178, 45.0]","[3.883025612848556, 3.294969675217061, 3.70219...",0.608834
151,13,126,90,0,0,43.4,0.583,42,1,"[13.0, 126.0, 90.0, 0.0, 0.0, 43.4, 0.583, 42.0]","[3.883025612848556, 3.91666206676745, 4.627745...",0.725763
152,13,145,82,19,110,22.2,0.245,57,0,"[13.0, 145.0, 82.0, 19.0, 110.0, 22.2, 0.245, ...","[3.883025612848556, 4.507269838740319, 4.21638...",0.607365


- the difference is that theres a new column called "scaled_features", which is the transformed (standardized) version of features. It is the version we feed into our ML model.

- the scaled features is actually better for the model because ScaledFeatures puts all feature columns on the same scale → the model treats both fairly and learns better weights

In [ ]:
from pyspark.sql import functions as func

prediction = prediction.withColumn('label2', func.round(prediction['label'], 2))
prediction=prediction.withColumn('prediction2', func.round(prediction['prediction'], 2))

In [ ]:
prediction.select('label2','prediction2').show()

+------+-----------+
|label2|prediction2|
+------+-----------+
|     0|       0.06|
|     1|       0.12|
|     0|      -0.04|
|     0|       0.11|
|     0|      -0.03|
|     0|       0.03|
|     1|        0.2|
|     0|       0.11|
|     1|       0.28|
|     0|       0.12|
|     0|       0.31|
|     1|       0.33|
|     0|       0.27|
|     1|       0.28|
|     0|       0.39|
|     1|       0.73|
|     0|       0.18|
|     0|       0.35|
|     1|       0.49|
|     1|       0.62|
+------+-----------+
only showing top 20 rows



# Evaluation

 show rmse, mae, r2 Save the best model

 To evaluate a PipelineModel in Spark MLlib, we need to use an evaluator.
  - RegressionEvaluator : Evaluator for Regression, which expects input columns prediction, label and an optional weight column.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# Evaluate the model
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(prediction)
mae = evaluator_mae.evaluate(prediction)
r2 = evaluator_r2.evaluate(prediction)

print("RMSE: %f" % rmse)
print("MAE: %f" % mae)
print("R2: %f" % r2)

RMSE: 0.388904
MAE: 0.326109
R2: 0.277989


# Save model

In [ ]:
pipeline.save("pipeline_model")

zip and download to local

In [ ]:
!zip -r pipeline_model.zip pipeline_model

  adding: pipeline_model/ (stored 0%)
  adding: pipeline_model/metadata/ (stored 0%)
  adding: pipeline_model/metadata/_SUCCESS (stored 0%)
  adding: pipeline_model/metadata/._SUCCESS.crc (stored 0%)
  adding: pipeline_model/metadata/.part-00000.crc (stored 0%)
  adding: pipeline_model/metadata/part-00000 (deflated 23%)
  adding: pipeline_model/stages/ (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/ (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/metadata/ (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/metadata/_SUCCESS (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/metadata/._SUCCESS.crc (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/metadata/.part-00000.crc (stored 0%)
  adding: pipeline_model/stages/0_VectorAssembler_5843436c067f/metadata/part-00000 (deflated 35%)
  adding: pipeline_model/stages/1_StandardScaler_097fb43b55b2/ (stored 0%)
  adding

In [ ]:
files.download("pipeline_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

save the predictions

In [ ]:
prediction.select("prediction", "label").write.save(path="predictions",
                                                     format="csv",
                                                     header='true',
                                                     mode="overwrite")

In [ ]:
sc.stop()